## Load package

In [ ]:
import os
from glob import glob
import re
import json

# output_dir = os.path.join('outputs','eda')
# output_dir = os.path.join('data','reviewed_labels')

# font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf
font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf

import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sns

from src.utils.preprocessing import Preprocessor

from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import Isomap, TSNE
from sklearn.cluster import DBSCAN
from sklearn.metrics import confusion_matrix, classification_report
from umap import UMAP



negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]


def negative_words():
    return ' ?no | ?neither | ?unlike | ?without | ?not? (sugges|compat)| ?negative | ?previous'


def uncertain_words():
    return ' ?rule out '


def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


def excel_loader(paths, target):
    paths = [p for p in paths if target in p]
    df = pd.concat(
        [column_selector(
            pd.read_excel(p, engine = 'openpyxl')).dropna().set_axis(['index','검사결과결론내용', 'label'], axis = 1) for p in paths])
    df = data_label_prepr(df)
    return df


def column_selector(df, fixed_col = ['index','검사결과결론내용']):
    # print([c for c in df.columns if c not in fixed_col][-1])
    return df.loc[:, fixed_col + [[c for c in df.columns if c not in fixed_col][-1]]]


def data_label_prepr(df):
    df['label'] = df['label'].apply(label_repr)
    return df


def label_repr(s):
    if 'recur' in s:
        return 'positive'
    elif 'region' in s:
        return 'RLN'
    elif 'meta' in s:
        return 'positive'
    elif 'positive' in s:
        return 'positive'
    elif 'x' in s:
        return 'negative'
    elif 'negative' in s:
        return 'negative'
    elif 'uncertain' in s:
        return 'uncertain'
    else:
        assert False, s


txt2label = {
    'meta' : 'positive',
    'regional LN metastasis' : 'negative',
    'recur' : 'positive',
    'uncertain' : 'uncertain',
}
textmapping_fn = lambda x: txt2label[x] if x in txt2label else 'negative'


def transfer_bianry_fn(df):
    df.loc[:, 'label'] = df['label'].apply(lambda x: 'positive' if x == 'positive' else 'negative')
    return df


font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용


# figdir = os.path.join(output_dir,'figures')
# filedir = os.path.join(output_dir,'files')
# os.makedirs(figdir, exist_ok=True)
# os.makedirs(filedir, exist_ok=True)


## Count total data 

In [ ]:
target = 'metas'

In [ ]:
print(glob(os.path.join('data','reports','*')))
df = pd.read_csv(glob(os.path.join('data','reports','*'))[0], encoding = 'utf-8-sig', engine = 'c')
text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)


In [ ]:
text_df[1]

In [ ]:
data_dict = preprocessor.target_filtering(target)

In [ ]:
merged_target_df = pd.concat([check_label(d, k) for k, d in data_dict.items()]).sort_index()
df.loc[merged_target_df.index, f'{target}_text'] = merged_target_df['검사결과결론내용']
merged_df = merged_target_df.copy()

In [ ]:
merged_df.shape, merged_df['검사결과결론내용'].drop_duplicates().shape

In [ ]:
log_dir = os.path.join('outputs','eda')
os.makedirs(log_dir, exist_ok = True)

## Count reivision data

In [ ]:
from collections import defaultdict

def revision_checker(rdf, drop_duplicates = False):

    novel_case = []
    unique_normal_case = []
    multiple_normal_case = []
    for t, sdf in rdf.groupby('검사결과결론내용'):
        if sdf.shape[0] == 1:
            unique_normal_case.append(sdf)
            continue

        unique_labels = sdf.label.unique()
        if len(unique_labels) == 1:
            multiple_normal_case.append(sdf)
            continue
        else: 
            novel_case.append(sdf)

    unique_df = pd.concat(unique_normal_case)
    
    if len(multiple_normal_case) > 0:
        multiple_df = pd.concat(multiple_normal_case)
        if drop_duplicates:
            multiple_df = multiple_df.drop_duplicates('검사결과결론내용')
        training_label_df = pd.concat([unique_df, multiple_df]).sort_values('index').reset_index(drop = True)
    else:
        training_label_df = unique_df.sort_values('index').reset_index(drop= True)

    novel_counter = defaultdict(int)
    for nc in novel_case:
        novel_counter['-'.join(sorted(nc.label.to_list()))] += 1 if drop_duplicates else nc.shape[0]

    return dict(
        label_df = training_label_df,
        unique_df = unique_df,
        multiple_df = multiple_df if len(multiple_normal_case) > 0 else None,
        novel_df = pd.concat(novel_case) if len(novel_case) > 0 else None,
        novel_counter = novel_counter
    )

In [ ]:

folder_names = ['2509*', '2510*']
label_orders = [
    ['positive','uncertain','negative'],
    ['positive','uncertain','RLN','negative']
]
targets = ['recur','metas']

totalc = []
total_ec = []
multiple_ratios = []
review_dict = dict()
for target, label_order in zip(targets, label_orders):
    reviews = []
    sole_reviews = []
    counter_df = []
    sole_counter_df = []
    review_dict[target] = dict()
    for folder_name in folder_names:
        data_dir_path = glob(os.path.join('data','reviewer',folder_name,'*'))
        print(data_dir_path)

        reviewed_df = excel_loader(data_dir_path, target)
        sole_reviewed_df = reviewed_df.drop_duplicates('검사결과결론내용')
        review_dict[target][folder_name] = dict(
            reviewed_df = reviewed_df,
            sole_reviewed_df = sole_reviewed_df
        )

        counter_df.append(reviewed_df.label.value_counts().loc[label_order])
        sole_counter_df.append(sole_reviewed_df.label.value_counts().loc[label_order])

        reviews.append(reviewed_df)
        sole_reviews.append(sole_reviewed_df)


    reviewed_df = pd.concat(reviews)
    return_info = revision_checker(reviewed_df.reset_index(drop = True))
    counter_df.append(return_info['label_df'].label.value_counts().loc[label_order])
    counter_df = pd.concat(counter_df, axis = 1).set_axis( ['1','2','1+2'], axis = 1)
    counter_df.loc['error','1+2'] = sum(return_info['novel_counter'].values())
    

    sole_reviewed_df = pd.concat(sole_reviews).drop_duplicates('검사결과결론내용')
    sole_return_info = revision_checker(sole_reviewed_df.reset_index(drop = True), drop_duplicates = True)
    sole_counter_df.append(sole_return_info['label_df'].label.value_counts().loc[label_order])
    sole_counter_df = pd.concat(sole_counter_df, axis = 1).set_axis( ['1','2','1+2'], axis = 1)
    sole_counter_df.loc['error','1+2'] = sum(sole_return_info['novel_counter'].values())

    concat_df = pd.concat([sole_counter_df, counter_df], axis = 1)
    concat_df.loc['total'] = concat_df.sum()
    totalc.append(concat_df)
    total_ec.append([sole_return_info, return_info])


    
    # temp_df = return_info['label_df']
    # temp_df = reviewed_df.copy()
    # index_counter = temp_df['index'].value_counts()
    # multiple_index = sorted(index_counter[index_counter >= 2].index.tolist())
    # multiple_dict = dict()
    # for n, (idx, sdf) in enumerate(temp_df.loc[temp_df['index'].isin(multiple_index)].groupby('검사결과결론내용'), 1):
    #     multiple_dict[n] = dict(
    #         index = idx,
    #         label = sdf.label.unique().tolist(),
    #         text = sdf['검사결과결론내용'].unique().tolist(),
    #         count = sdf.shape[0],
    #     )

#     # pass
#     multiple_labels = [i['label'] for i in multiple_dict.values()]
#     multiple_error_ratio = len([i for i in multiple_labels if len(i) > 1])/len(multiple_labels) * 100
#     print(len(multiple_labels), multiple_error_ratio)
#     print(sum([i['count'] for i in multiple_dict.values()]) - len(multiple_labels))

pd.concat(totalc, axis = 0).to_excel(os.path.join(log_dir, 'data-count-revision.xlsx'))

In [ ]:

nt = 0
accs = []
for nt in range(len(total_ec)):
    return_info = total_ec[nt][1] # only unique case test that we can consider twice revision
    multiple_df = return_info['multiple_df']
    unique_df = return_info['unique_df']
    novel_df = return_info['novel_df']

    temp_df = pd.concat([multiple_df, novel_df])
    length = len(temp_df)
    temp_df = temp_df.groupby('검사결과결론내용').agg(lambda x: x.unique().tolist())
    unique_length = len(temp_df)
    acc = temp_df.label.apply(lambda x: len(x) > 1).sum() / unique_length * 100
    accs.append([length, unique_length, acc])
print(accs)

In [ ]:
100 - 96.43, 100 - 92.71

In [ ]:
nt = 1
return_info = total_ec[nt][1]
multiple_df = return_info['multiple_df']
unique_df = return_info['unique_df']
novel_df = return_info['novel_df']

# multiple_df

n_unique = unique_df.shape[0]
n_multiple = multiple_df.shape[0] if multiple_df is not None else 0
n_novel = novel_df.shape[0] if novel_df is not None else 0

temp_df = pd.concat([multiple_df, novel_df]).drop_duplicates('검사결과결론내용')

tc = totalc[nt]
tc_delta = int(tc.iloc[-1,5] - tc.iloc[-1,2] - tc.iloc[-2,-1])
display(tc.fillna(0).astype(int))
print(n_unique, n_multiple, n_unique + n_multiple, tc_delta, n_unique + n_multiple - tc_delta)


## Count mismatched label by each revision

In [ ]:
# target = 'metas'

review_df_type = 'sole_reviewed_df'
# review_df_type = 'reviewed_df'

for review_df_type in ['sole_reviewed_df','reviewed_df']:
    save_data_xlsx = []
    for target in ['recur','metas']:


        lo = label_orders[0] if target == 'recur' else label_orders[1]

        sole_2509 = review_dict[target]['2509*'][review_df_type]
        sole_2510 = review_dict[target]['2510*'][review_df_type]

        common_indices = set(sole_2509['index']).intersection(set(sole_2510['index']))
        common_indices = list(common_indices)
        print(target)
        print(f"Number of overlapping indices: {len(common_indices)}")

        # If you want to see which rows overlap:
        overlap_2509 = sole_2509[sole_2509['index'].isin(common_indices)]
        overlap_2510 = sole_2510[sole_2510['index'].isin(common_indices)]

        overlap_df = overlap_2509.merge(overlap_2510, on = ['index','검사결과결론내용'], how = 'inner')
        error_idx = (overlap_df.label_x != overlap_df.label_y)
        print(f"Number of error indices: {error_idx.sum()}")
        print(f'Percentage of error indices: {error_idx.sum() / len(common_indices) * 100:.3f}%')

        exclude_2509 = sole_2509[~sole_2509['index'].isin(common_indices)]
        exclude_2510 = sole_2510[~sole_2510['index'].isin(common_indices)]

        # display(exclude_2509.sort_values('index').label.value_counts().loc[lo] + overlap_2509.label.value_counts().loc[lo])
        # display(exclude_2510.sort_values('index').label.value_counts().loc[lo] + overlap_2510.label.value_counts().loc[lo])


        rdf_index = pd.MultiIndex.from_product([['Revision 1','Revision 2',],['Uni.','D.C.','Sum']])
        rdf = pd.concat([
            exclude_2509.sort_values('index').label.value_counts().loc[lo],
            overlap_2509.label.value_counts().loc[lo],
            exclude_2509.sort_values('index').label.value_counts().loc[lo] +  overlap_2509.label.value_counts().loc[lo],
            exclude_2510.sort_values('index').label.value_counts().loc[lo],
            overlap_2510.label.value_counts().loc[lo],
            exclude_2510.sort_values('index').label.value_counts().loc[lo] +  overlap_2510.label.value_counts().loc[lo],
        ], axis = 1)

        rdf.columns = rdf_index
        rdf.loc['Sum',:] = rdf.sum()
        rdf = rdf.astype(int)

        display(rdf)
        save_data_xlsx.append(rdf)
    pd.concat(save_data_xlsx, axis = 0).to_excel(
        os.path.join(log_dir, f'data-count-{review_df_type}-revision-nodrop-error.xlsx')
    )

## Plot label transition

In [ ]:
review_df_type = 'sole_reviewed_df'
# review_df_type = 'reviewed_df'
for review_df_type in ['sole_reviewed_df','reviewed_df']:
    for target in ['recur','metas']:
        target_name = 'Recurrence' if target == 'recur' else 'Metastasis'

        lo = label_orders[0] if target == 'recur' else label_orders[1]

        sole_2509 = review_dict[target]['2509*'][review_df_type]
        sole_2510 = review_dict[target]['2510*'][review_df_type]

        common_indices = set(sole_2509['index']).intersection(set(sole_2510['index']))
        common_indices = list(common_indices)
        print(target)
        print(f"Number of overlapping indices: {len(common_indices)}")

        # If you want to see which rows overlap:
        overlap_2509 = sole_2509[sole_2509['index'].isin(common_indices)]
        overlap_2510 = sole_2510[sole_2510['index'].isin(common_indices)]

        overlap_df = overlap_2509.merge(overlap_2510, on = ['index','검사결과결론내용'], how = 'inner')
        error_idx = (overlap_df.label_x != overlap_df.label_y)
        print(f"Number of error indices: {error_idx.sum()}")
        print(f'Percentage of error indices: {error_idx.sum() / len(common_indices) * 100:.4f}%')

        order = lo
        # confusion matrix 생성
        conf_matrix = pd.crosstab(
            overlap_df.loc[error_idx, 'label_x'],
            overlap_df.loc[error_idx, 'label_y'],
            rownames=['label_x'], 
            colnames=['label_y']
        ).reindex(index=order, columns=order, fill_value=0)

        order = [i.capitalize() if i != 'RLN' else 'Regional' for i in order ]
        plt.rcParams['font.size'] = 14
        plt.figure(figsize=(6,5))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=order, yticklabels=order, cbar = False)
        plt.title(f'{target_name}: Revision 1 → Revision 2')
        plt.ylabel('Revision 1')
        plt.xlabel('Revision 2')
        # plt.show()
        plt.tight_layout()
        plt.savefig(os.path.join(log_dir, f'data-count-{review_df_type}-{target}-revision-confusion-matrix.png'))

## Grep the mismatched label cases

In [ ]:
# review_df_type = 'sole_reviewed_df'
# review_df_type = 'reviewed_df'
for review_df_type in ['sole_reviewed_df','reviewed_df']:
    for target in ['recur','metas']:
        target_name = 'Recurrence' if target == 'recur' else 'Metastasis'

        lo = label_orders[0] if target == 'recur' else label_orders[1]

        sole_2509 = review_dict[target]['2509*'][review_df_type]
        sole_2510 = review_dict[target]['2510*'][review_df_type]

        common_indices = set(sole_2509['index']).intersection(set(sole_2510['index']))
        common_indices = list(common_indices)
        print(target)
        print(f"Number of overlapping indices: {len(common_indices)}")

        # If you want to see which rows overlap:
        overlap_2509 = sole_2509[sole_2509['index'].isin(common_indices)]
        overlap_2510 = sole_2510[sole_2510['index'].isin(common_indices)]

        overlap_df = overlap_2509.merge(overlap_2510, on = ['index','검사결과결론내용'], how = 'inner')
        error_idx = (overlap_df.label_x != overlap_df.label_y)
        print(f"Number of error indices: {error_idx.sum()}")
        print(f'Percentage of error indices: {error_idx.sum() / len(common_indices) * 100:.3f}%')

        error_df = overlap_df[error_idx]
        error_df.to_excel(os.path.join(log_dir, f'data-count-{review_df_type}-{target}-revision-error.xlsx'))


In [ ]:
review_df_type = 'sole_reviewed_df'
# review_df_type = 'reviewed_df'
# for review_df_type in ['sole_reviewed_df','reviewed_df']:
for target in ['recur','metas']:
    target_name = 'Recurrence' if target == 'recur' else 'Metastasis'

    lo = label_orders[0] if target == 'recur' else label_orders[1]

    sole_2509 = transfer_bianry_fn(review_dict[target]['2509*'][review_df_type])
    sole_2510 = transfer_bianry_fn(review_dict[target]['2510*'][review_df_type])

    common_indices = set(sole_2509['index']).intersection(set(sole_2510['index']))
    common_indices = list(common_indices)
    print(target)
    print(f"Number of overlapping indices: {len(common_indices)}")

    # If you want to see which rows overlap:
    overlap_2509 = sole_2509[sole_2509['index'].isin(common_indices)]
    overlap_2510 = sole_2510[sole_2510['index'].isin(common_indices)]

    overlap_df = overlap_2509.merge(overlap_2510, on = ['index','검사결과결론내용'], how = 'inner')
    error_idx = (overlap_df.label_x != overlap_df.label_y)
    print(f"Number of error indices: {error_idx.sum()}")
    print(f'Percentage of error indices: {error_idx.sum() / len(common_indices) * 100:.3f}%')

    error_df = overlap_df[error_idx]
    # error_df.to_excel(os.path.join(log_dir, f'data-count-{review_df_type}-{target}-revision-error.xlsx'))
